In [ ]:
using Bessels, Plots

const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
φ_b = 0.204

E_J = 37.12*2pi/2

#Environment
κ_b=5/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.1
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 4*g2

g = E_J*sin(ε_p)*φ_a^2*φ_b
params = (g,ε_d,κ_b);

M = 100

A_0 = sqrt(ε_d / g)
B_0 = sqrt(2) * ε_d / κ_b

In [ ]:
κ_b^2/4/ε_d/g

In [ ]:
eps = 1e-1
ϕ_a = [+eps, -eps]
ϕ_b = [+eps, -B_0+eps]

# initial string in phase-space
Δs = 1 / M
s=0:Δs:1
T = 10000
Δt = T/M

A_init =@. ϕ_a[1] + (ϕ_b[1]-ϕ_a[1])*s
B_init =@. ϕ_a[2] + (ϕ_b[2]-ϕ_a[2])*s
P_A_init = eps*zeros((M+1))
P_B_init = eps*zeros((M+1))


ϕ_init = hcat(A_init, B_init)
θ_init = hcat(P_A_init, P_B_init);


In [ ]:
α = 1
Δτ = 1e-2
N_iter = 10

u = copy(ϕ_init .+ α .* θ_init)
v = copy(ϕ_init .- α .* θ_init)
ϕ = (u .+ v) ./ 2
θ = (u .- v) ./ (2α);

In [ ]:
for n in 1:N_iter
    # =========================================================
    # PASS 1: build lambda(u^n,v^n), update u from right boundary
    # =========================================================
    gradθ = zeros(M + 1, 2)
    gradϕ = zeros(M + 1, 2)
    Hvals = zeros(M + 1)

    for i in 1:M+1
        A = ϕ[i, 1]
        B = ϕ[i, 2]
        P_A = θ[i, 1]
        P_B = θ[i, 2]

        gradϕ[i, 1] = sqrt(2)*g*(A*P_B-P_A*B)
        gradϕ[i, 2] = -sqrt(2)*g*A*P_A-κ_b*P_B/2

        gradθ[i, 1] = sqrt(2)*g*(P_A*P_B-A*B)
        gradθ[i, 2] = g*(A^2+P_A^2)/sqrt(2)-ε_d/sqrt(2)-κ_b*B/2-κ_b*P_B

        Hvals[i] = -sqrt(2)*g*A*B*P_A+(g*(A^2+P_A^2)/sqrt(2)-ε_d/sqrt(2)-κ_b*B/2)*P_B-κ_b*P_B^2/2
    end

    fvals = α .* gradϕ .- gradθ

    u_new = similar(u)

    # right boundary condition
    u_new[end, :] .= -v[end, :] .+ 2 .* ϕ_b

    # first-order interior sweep
    for i in M:-1:1
        u_new[i, :] .= (u[i, :] .+ u_new[i+1, :])/2 .+ Δt .* fvals[i,:]/2
    end


    # =========================================================
    # PASS 2: build lambda(u^{n+1},v^n), update v from left boundary
    # =========================================================
    # reconstruct phi,theta after U update
    ϕ = (u_new .+ v) ./ 2
    θ = (u_new .- v) ./ (2α)

    ϕ[1, :] .= ϕ_init[1, :]
    ϕ[end, :] .= ϕ_init[end, :]
    θ[1, :] .= θ_init[1, :]
    θ[end, :] .= θ_init[end, :]

    gradθ .= 0.0
    gradϕ .= 0.0
    Hvals .= 0.0

    for i in 1:M+1
        A = ϕ[i, 1]
        B = ϕ[i, 2]
        P_A = θ[i, 1]
        P_B = θ[i, 2]

        gradϕ[i, 1] = sqrt(2)*g*(A*P_B-P_A*B)
        gradϕ[i, 2] = -sqrt(2)*g*A*P_A-κ_b*P_B/2

        gradθ[i, 1] = sqrt(2)*g*(P_A*P_B-A*B)
        gradθ[i, 2] = g*(A^2+P_A^2)/sqrt(2)-ε_d/sqrt(2)-κ_b*B/2-κ_b*P_B

        Hvals[i] = -sqrt(2)*g*A*B*P_A+(g*(A^2+P_A^2)/sqrt(2)-ε_d/sqrt(2)-κ_b*B/2)*P_B-κ_b*P_B^2/2
    end

    gvals = α .* gradϕ .+ gradθ

    v_new = similar(v)

    # left boundary condition
    v_new[1, :] .= -u_new[1, :] .+ 2 .* ϕ_a

    # first-order interior sweep
    for i in 2:M+1
        v_new[i, :] .= (v[i, :] .+ v_new[i-1, :])/2 .+ Δt .* gvals[i,:]/2

    end

    # reconstruct phi,theta after V update
    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2α)

    ϕ[1, :] .= ϕ_init[1, :]
    ϕ[end, :] .= ϕ_init[end, :]
    θ[1, :] .= θ_init[1, :]
    θ[end, :] .= θ_init[end, :]

end

In [ ]:
A = ϕ[:, 1]
B = ϕ[:, 2]
P_A = θ[:, 1]
P_B = θ[:, 2]

# --- subplot 1 ---
p1 = scatter(P_A, A, ms=.5,
    xlabel="P_A",
    ylabel="A",
     label="GDA")

# --- subplot 2 ---
p2 = scatter(P_B, B, ms=.5,
    xlabel="P_B",
    ylabel="B",
    label="GDA")

# --- subplot 3 ---
p3 = scatter(A, B, ms=.5,
    xlabel="A",
    ylabel="B",
    label="GDA")

# --- combine ---
plot(p1, p2, p3, layout=(1,3))